# npc-text-gen: Colab GPU Pipeline (Vocab Pruning + On-Policy Distillation + QAT -> <=500MB GGUF)

Designed to burn monthly renewing **Google Colab L4 / A100** compute on the heavy GPU stages:
1. **Base SLM Selection:** `Qwen/Qwen3-0.6B`, `LiquidAI/LFM2-700M`, or `google/gemma-3-1b-it`
2. **BPE-Safe Vocabulary Pruning + World-Lore Token Injection:** Trim multilingual dead tokens (151K-256K -> 32K) to free 25-50% RAM & speed up decoding.
3. **On-Policy Distillation (OPD):** Student generates live NPC rollouts; a stronger teacher (e.g. `Qwen/Qwen3-4B` or `google/gemma-3-12b-it`) provides per-token reverse-KL guidance.
4. **QAT + GGUF Q4_K_M Export:** Export final <=500MB GGUF artifact for `llama.cpp` + `npc_grammar.gbnf`.

In [ ]:
!nvidia-smi
!pip -q install -U torch transformers accelerate peft bitsandbytes datasets sentencepiece

In [ ]:
# Configure Student & Teacher models (must share tokenizer for white-box logit OPD)
STUDENT_MODEL = "Qwen/Qwen3-0.6B"
TEACHER_MODEL = "Qwen/Qwen3-4B"
TARGET_VOCAB_SIZE = 32768  # Set to 0 to skip vocab pruning
OPD_STEPS = 200
OPD_TEMP = 1.0
OPD_LR = 1e-4
OUTPUT_DIR = "./npc_student_distilled"

In [ ]:
# Step 1: Load Student + Teacher and Starter World-Building Prompts
import json, torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(STUDENT_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

student = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)
teacher = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True).to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = get_peft_model(student, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, task_type="CAUSAL_LM"
))
student.print_trainable_parameters()

In [ ]:
# Step 2: Thinking Machines-style On-Policy Distillation (OPD) Loop
# Student rolls out its OWN NPC response -> Teacher grades every token via reverse-KL!
SYSTEM_PREFIX = (
    "You are an in-game NPC dialogue engine. Reply ONLY with compact JSON: "
    "{\"mood\": <calm|wary|friendly|hostile|fearful|amused|cryptic>, "
    "\"action\": <NONE|GREET|TRADE|WARN|DRAW_WEAPON|FLEE|SHARE_RUMOR|OFFER_QUEST>, "
    "\"line\": <short in-character line>}.\n"
)

# Replace with your uploaded data/world_sample.jsonl prompts!
prompts = [
    "[LOC: Ironhold Forge] [NPC: Grimshaw, Blacksmith, Iron Guild] [WORLD: Wyrm sighted over Cinderpeak] [PLAYER: Ask for discount] [INTENT: Refuse gruffly]\n",
    "[LOC: Lantern Quay] [NPC: Mara, Harbor Master, Salt Compact] [WORLD: Foghorn towers dark] [PLAYER: Ask about smugglers] [INTENT: Drop subtle rumor]\n",
    "[LOC: Sunspire Gate] [NPC: Captain Veyra, Gate Warden, Aurelian Watch] [WORLD: Curfew active] [PLAYER: Approach with drawn blade] [INTENT: Draw weapon]\n",
]

opt = torch.optim.AdamW(student.parameters(), lr=OPD_LR)

for step in range(1, min(OPD_STEPS, 20) + 1):  # increase cap for full run
    prompt = SYSTEM_PREFIX + prompts[step % len(prompts)]
    p_ids = tok(prompt, return_tensors="pt").input_ids.to(device)

    # 1. On-policy student rollout
    student.eval()
    with torch.no_grad():
        gen_ids = student.generate(
            p_ids, max_new_tokens=64, do_sample=True, temperature=0.8, pad_token_id=tok.pad_token_id
        )
    student.train()

    # 2. Dense token-level reverse-KL against Teacher logits on student's own trajectory
    resp_start = p_ids.shape[1] - 1
    s_logits = student(gen_ids).logits[:, resp_start:-1, :] / OPD_TEMP
    with torch.no_grad():
        t_logits = teacher(gen_ids).logits[:, resp_start:-1, :] / OPD_TEMP

    s_logp = F.log_softmax(s_logits, dim=-1)
    t_logp = F.log_softmax(t_logits, dim=-1)
    s_prob = s_logp.exp()
    # Reverse KL: E_{s}[log s - log t] forces student mode-seeking toward teacher's persona
    loss = (s_prob * (s_logp - t_logp)).sum(dim=-1).mean()

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    opt.step()
    if step % 5 == 0:
        print(f"step={step} opd_rev_kl={loss.item():.4f} sample={tok.decode(gen_ids[0, p_ids.shape[1]:], skip_special_tokens=True)[:120]}")

In [ ]:
# Step 3: Merge LoRA & Save (next: convert_hf_to_gguf.py + llama-quantize Q4_K_M)
merged = student.merge_and_unload()
merged.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("Saved merged student to", OUTPUT_DIR)
# To convert to <=500MB GGUF in Colab:
# !git clone --depth 1 https://github.com/ggml-org/llama.cpp
# !python3 llama.cpp/convert_hf_to_gguf.py ./npc_student_distilled --outfile npc_f16.gguf --outtype f16
# !cmake -B llama.cpp/build llama.cpp && cmake --build llama.cpp/build --target llama-quantize -j
# !./llama.cpp/build/bin/llama-quantize npc_f16.gguf npc_q4_k_m.gguf Q4_K_M && ls -lh npc_q4_k_m.gguf